# Práctica · Spark DataFrames · Clientes y pedidos

Completa las celdas indicadas. Trabaja sobre el entorno dockerizado incluido en `spark_jupyter/`.

In [1]:
from iniciar_spark import get_spark
from pyspark.sql import functions as F

spark = get_spark("PracticaClientesPedidos")
print("SparkSession creada")
print("Versión de Spark:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 17:18:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession creada
Versión de Spark: 4.1.1


In [2]:
from pathlib import Path

data_dir = Path("/opt/spark-apps/datos")
clientes_path = data_dir / "clientes.csv"
pedidos_path = data_dir / "pedidos.csv"

print("Ruta clientes:", clientes_path)
print("Ruta pedidos:", pedidos_path)


Ruta clientes: /opt/spark-apps/datos/clientes.csv
Ruta pedidos: /opt/spark-apps/datos/pedidos.csv


## 1. Lee ambos CSV y muestra su esquema y algunas filas.

In [3]:
# TODO
ruta_clientes = "/opt/spark-apps/datos/clientes.csv"
ruta_pedidos = "/opt/spark-apps/datos/pedidos.csv"

df_clientes = spark.read.csv(
    ruta_clientes,
    header=True,
    inferSchema=True,
    sep=";"
)

df_pedidos = spark.read.csv(
    ruta_pedidos,
    header=True,
    inferSchema=True,
    sep=";"
)

# Filas de ejemplo.
print("==== FILAS ====") 
print("CLIENTES ") 
df_clientes.show()
print("\nPEDIDOS ") 
df_pedidos.show()

# Esquema
print("\n==== ESQUEMAS ====") 
df_clientes.printSchema()
df_pedidos.printSchema()

==== FILAS ====
CLIENTES 
+----------+------+----------+--------+
|id_cliente|nombre|    ciudad|segmento|
+----------+------+----------+--------+
|         1|   Ana|  Sevilla |Estandar|
|         2|  Luis|    Bilbao| Premium|
|         3| Marta| Alicante |Estandar|
|         4| Pablo|   Madrid | Premium|
|         5| Lucia|    Bilbao| Premium|
|         6|Carlos| Alicante |Estandar|
|         7| Elena|    Bilbao|Estandar|
|         8|Javier|    Madrid| Premium|
|         9|  Sara|    Murcia| Premium|
|        10| David|    Bilbao|Estandar|
|        11|Raquel|  Sevilla | Premium|
|        12|Miguel|  Zaragoza|Estandar|
|        13| Irene|    Madrid| Premium|
|        14|Sergio|   Murcia |Estandar|
|        15| Paula|   Granada|Estandar|
|        16| Diego|  Granada | Premium|
|        17|  Alba|    Madrid|Estandar|
|        18| Jorge|   Sevilla| Premium|
|        19| Clara|    Murcia|Estandar|
|        20|Adrian|  Valencia| Premium|
+----------+------+----------+--------+
only showing t

## 2. Limpia el DataFrame de clientes: trim en ciudad y dropDuplicates.

In [4]:
# ELIMINAR FILAS DUPLICADAS
print("ELIMINAR FILAS DUPLICAS")

# 1. Número de filas que tiene el DataFrame original de clientes
print("\n=== Número de filas de clientes ===")
print("Filas en clientes:", df_clientes.count())

# 2. Mostramos una muestra de los datos para inspección visual
print("\n=== Tabla clientes ===")
df_clientes.show(10, truncate=False)

# 3. Guardamos el número total de filas 
total_clientes = df_clientes.count()

# 4. Calculamos cuántas filas quedarían al eliminar duplicados 
clientes_sin_duplicados = df_clientes.dropDuplicates().count()

# 5. Mostramos el total de filas y cuántos duplicados se han detectado
print("\n=== Comprobación de duplicados ===")
print("Total filas clientes:", total_clientes)
print("Duplicados detectados:", total_clientes - clientes_sin_duplicados)

# 6. Creamos un nuevo DataFrame sin filas duplicadas
df_clientes_limpio = df_clientes.dropDuplicates()

# 7. Comprobamos el resultado tras la limpieza de duplicados
print("\n=== Resultado tras eliminar duplicados ===")
print("Filas en clientes original:", df_clientes.count())
print("Filas en clientes sin duplicados:", df_clientes_limpio.count())


ELIMINAR FILAS DUPLICAS

=== Número de filas de clientes ===
Filas en clientes: 43

=== Tabla clientes ===
+----------+------+----------+--------+
|id_cliente|nombre|ciudad    |segmento|
+----------+------+----------+--------+
|1         |Ana   | Sevilla  |Estandar|
|2         |Luis  |Bilbao    |Premium |
|3         |Marta | Alicante |Estandar|
|4         |Pablo | Madrid   |Premium |
|5         |Lucia |Bilbao    |Premium |
|6         |Carlos| Alicante |Estandar|
|7         |Elena |Bilbao    |Estandar|
|8         |Javier|Madrid    |Premium |
|9         |Sara  |Murcia    |Premium |
|10        |David |Bilbao    |Estandar|
+----------+------+----------+--------+
only showing top 10 rows

=== Comprobación de duplicados ===
Total filas clientes: 43
Duplicados detectados: 3

=== Resultado tras eliminar duplicados ===
Filas en clientes original: 43
Filas en clientes sin duplicados: 40


In [6]:
# ELIMINAR NULOS
print("\nELIMINAR VALORES NULOS")

from pyspark.sql.functions import col, sum, when

# 1. Contamos cuántos valores nulos hay en cada columna de clientes
print("\n=== Valores nulos en clientes ===")
df_clientes_limpio.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_clientes_limpio.columns
]).show()

# 2. Contamos cuántos valores nulos hay en cada columna de pedidos
print("=== Valores nulos en pedidos ===")
df_pedidos.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_pedidos.columns
]).show()

# 3. Mostramos cuántas filas tiene pedidos antes de limpiar nulos
print("=== Pedidos antes de limpiar nulos ===")
print("Filas en pedidos:", df_pedidos.count())

# 4. Eliminamos filas con nulos en las columnas críticas para el análisis
df_pedidos_limpio = df_pedidos.dropna(subset=["producto", "cantidad"])

# 5. Mostramos cuántas filas quedan después de limpiar
print("\n=== Pedidos después de limpiar nulos ===")
print("Filas en pedidos sin nulos:", df_pedidos_limpio.count())
print("Filas eliminadas:", df_pedidos.count() - df_pedidos_limpio.count())


ELIMINAR VALORES NULOS

=== Valores nulos en clientes ===
+----------+------+------+--------+
|id_cliente|nombre|ciudad|segmento|
+----------+------+------+--------+
|         0|     0|     0|       0|
+----------+------+------+--------+

=== Valores nulos en pedidos ===
+---------+----------+-----+--------+--------+---------------+
|id_pedido|id_cliente|fecha|producto|cantidad|precio_unitario|
+---------+----------+-----+--------+--------+---------------+
|        0|         0|    0|       2|       3|              0|
+---------+----------+-----+--------+--------+---------------+

=== Pedidos antes de limpiar nulos ===
Filas en pedidos: 120

=== Pedidos después de limpiar nulos ===
Filas en pedidos sin nulos: 115
Filas eliminadas: 5


In [7]:
# ELIMINAR ESPACIOS
print("\nELIMINAR ESPACIOS")

from pyspark.sql.functions import trim, col

# 1. Limpiamos espacios al inicio y al final en las columnas de texto
df_clientes_limpio = df_clientes_limpio.withColumn("nombre", trim(col("nombre"))) \
                                       .withColumn("ciudad", trim(col("ciudad"))) \
                                       .withColumn("segmento", trim(col("segmento")))

# 2. De pedido
df_pedidos_limpio = df_pedidos_limpio.withColumn("producto", trim(col("producto")))

# 3. Mostramos una muestra después de la limpieza
print("\n=== Muestra después de limpiar espacios ===")
df_clientes_limpio.show(10, truncate=False)
df_pedidos_limpio.show(10, truncate=False)



ELIMINAR ESPACIOS

=== Muestra después de limpiar espacios ===
+----------+--------+--------+--------+
|id_cliente|nombre  |ciudad  |segmento|
+----------+--------+--------+--------+
|18        |Jorge   |Sevilla |Premium |
|40        |Tomas   |Valencia|Estandar|
|5         |Lucia   |Bilbao  |Premium |
|6         |Carlos  |Alicante|Estandar|
|14        |Sergio  |Murcia  |Estandar|
|8         |Javier  |Madrid  |Premium |
|17        |Alba    |Madrid  |Estandar|
|4         |Pablo   |Madrid  |Premium |
|2         |Luis    |Bilbao  |Premium |
|33        |Patricia|Alicante|Premium |
+----------+--------+--------+--------+
only showing top 10 rows
+---------+----------+----------+-----------+--------+---------------+
|id_pedido|id_cliente|fecha     |producto   |cantidad|precio_unitario|
+---------+----------+----------+-----------+--------+---------------+
|1001     |12        |2025-03-05|Ratón      |6.0     |16             |
|1002     |32        |2025-02-02|Ratón      |3.0     |19           

In [8]:
# RESULTADO FINAL
print("\n=== TABLAS LIMPIADAS ===")
df_clientes_limpio.show(10, truncate=False)
df_pedidos_limpio.show(10, truncate=False)


=== TABLAS LIMPIADAS ===
+----------+--------+--------+--------+
|id_cliente|nombre  |ciudad  |segmento|
+----------+--------+--------+--------+
|18        |Jorge   |Sevilla |Premium |
|40        |Tomas   |Valencia|Estandar|
|5         |Lucia   |Bilbao  |Premium |
|6         |Carlos  |Alicante|Estandar|
|14        |Sergio  |Murcia  |Estandar|
|8         |Javier  |Madrid  |Premium |
|17        |Alba    |Madrid  |Estandar|
|4         |Pablo   |Madrid  |Premium |
|2         |Luis    |Bilbao  |Premium |
|33        |Patricia|Alicante|Premium |
+----------+--------+--------+--------+
only showing top 10 rows
+---------+----------+----------+-----------+--------+---------------+
|id_pedido|id_cliente|fecha     |producto   |cantidad|precio_unitario|
+---------+----------+----------+-----------+--------+---------------+
|1001     |12        |2025-03-05|Ratón      |6.0     |16             |
|1002     |32        |2025-02-02|Ratón      |3.0     |19             |
|1003     |5         |2025-03-07|M

In [9]:
# VALIDAR TIPOS DE DATOS
print("\nVALIDAR TIPOS DE DATOS")

# Mostramos el esquema final de clientes tras la limpieza
print("\n=== Esquema de clientes limpio ===")
df_clientes_limpio.printSchema()

# Mostramos el esquema final de pedidos tras la limpieza
print("\n=== Esquema de pedidos limpio ===")
df_pedidos_limpio.printSchema()


VALIDAR TIPOS DE DATOS

=== Esquema de clientes limpio ===
root
 |-- id_cliente: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- segmento: string (nullable = true)


=== Esquema de pedidos limpio ===
root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: double (nullable = true)
 |-- precio_unitario: integer (nullable = true)



## 3. Convierte tipos en pedidos y crea la columna `importe`.

In [13]:
from pyspark.sql.functions import col

# TRANSFORMACIÓN
print("\nTRANSFORMACIÓN")

# Creamos una nueva columna llamada importe
df_pedidos_transformado = df_pedidos_limpio.withColumn(
    "importe", col("cantidad") * col("precio_unitario")
)

# Mostramos una muestra para comprobar el resultado
print("\n=== Pedidos con la nueva columna importe ===")
df_pedidos_transformado.show(10, truncate=False)




TRANSFORMACIÓN

=== Pedidos con la nueva columna importe ===
+---------+----------+----------+-----------+--------+---------------+-------+
|id_pedido|id_cliente|fecha     |producto   |cantidad|precio_unitario|importe|
+---------+----------+----------+-----------+--------+---------------+-------+
|1001     |12        |2025-03-05|Ratón      |6.0     |16             |96.0   |
|1002     |32        |2025-02-02|Ratón      |3.0     |19             |57.0   |
|1003     |5         |2025-03-07|Monitor    |2.0     |210            |420.0  |
|1004     |13        |2025-03-18|Impresora  |4.0     |205            |820.0  |
|1005     |41        |2025-03-10|Teclado    |5.0     |40             |200.0  |
|1007     |27        |2025-02-13|Ratón      |1.0     |23             |23.0   |
|1008     |7         |2025-02-04|Webcam     |6.0     |77             |462.0  |
|1009     |18        |2025-03-02|Teclado    |1.0     |38             |38.0   |
|1010     |31        |2025-02-14|Webcam     |1.0     |78             

## 4. Haz un join entre clientes y pedidos.

In [15]:
# INTEGRACIÓN DE DATOS
print("\nINTEGRACIÓN DE DATOS")

# Hacemos un join entre clientes y pedidos usando id_cliente
df_clientes_pedidos = df_pedidos_transformado.join(
    df_clientes_limpio,
    on="id_cliente",
    how="inner"
)

# Mostramos una muestra del resultado
print("\n=== Resultado del join entre pedidos y clientes ===")
df_clientes_pedidos.show(10, truncate=False)



INTEGRACIÓN DE DATOS

=== Resultado del join entre pedidos y clientes ===
+----------+---------+----------+-----------+--------+---------------+-------+-------+--------+--------+
|id_cliente|id_pedido|fecha     |producto   |cantidad|precio_unitario|importe|nombre |ciudad  |segmento|
+----------+---------+----------+-----------+--------+---------------+-------+-------+--------+--------+
|12        |1001     |2025-03-05|Ratón      |6.0     |16             |96.0   |Miguel |Zaragoza|Estandar|
|32        |1002     |2025-02-02|Ratón      |3.0     |19             |57.0   |Andres |Zaragoza|Premium |
|5         |1003     |2025-03-07|Monitor    |2.0     |210            |420.0  |Lucia  |Bilbao  |Premium |
|13        |1004     |2025-03-18|Impresora  |4.0     |205            |820.0  |Irene  |Madrid  |Premium |
|27        |1007     |2025-02-13|Ratón      |1.0     |23             |23.0   |Laura  |Bilbao  |Estandar|
|7         |1008     |2025-02-04|Webcam     |6.0     |77             |462.0  |Elena  

In [16]:
# Comparamos el número de pedidos antes y después del join
print("\n=== Comparación antes y después del join ===")

total_pedidos_antes = df_pedidos_transformado.count()
total_pedidos_despues = df_clientes_pedidos.count()

print("Pedidos antes del join:", total_pedidos_antes)
print("Pedidos después del join:", total_pedidos_despues)
print("Pedidos que no encontraron cliente y se perdieron:", total_pedidos_antes - total_pedidos_despues)


# Mostramos los pedidos que no tienen cliente asociado en la tabla de clientes
print("\n=== Pedidos sin cliente asociado ===")

pedidos_sin_cliente = df_pedidos_transformado.join(
    df_clientes_limpio,
    on="id_cliente",
    how="left_anti"
)

pedidos_sin_cliente.show(truncate=False)


=== Comparación antes y después del join ===
Pedidos antes del join: 115
Pedidos después del join: 106
Pedidos que no encontraron cliente y se perdieron: 9

=== Pedidos sin cliente asociado ===
+----------+---------+----------+-----------+--------+---------------+-------+
|id_cliente|id_pedido|fecha     |producto   |cantidad|precio_unitario|importe|
+----------+---------+----------+-----------+--------+---------------+-------+
|41        |1005     |2025-03-10|Teclado    |5.0     |40             |200.0  |
|46        |1022     |2025-02-27|Teclado    |3.0     |45             |135.0  |
|41        |1041     |2025-02-23|Teclado    |6.0     |18             |108.0  |
|48        |1058     |2025-03-17|Auriculares|3.0     |34             |102.0  |
|45        |1060     |2025-03-11|Impresora  |4.0     |193            |772.0  |
|42        |1074     |2025-02-16|Impresora  |2.0     |115            |230.0  |
|47        |1107     |2025-02-12|Monitor    |4.0     |204            |816.0  |
|46        |110

## 5. Filtra ventas Premium con importe >= 100.

In [17]:
# ANÁLISIS
print("\nANÁLISIS")

# Filtramos los pedidos con importe mayor de 100
pedidos_importe_alto = df_clientes_pedidos.filter(col("importe") > 100)

# Mostramos una muestra del resultado
print("\n=== Pedidos con importe mayor de 100 ===")
pedidos_importe_alto.show(10, truncate=False)


ANÁLISIS

=== Pedidos con importe mayor de 100 ===
+----------+---------+----------+---------+--------+---------------+-------+------+--------+--------+
|id_cliente|id_pedido|fecha     |producto |cantidad|precio_unitario|importe|nombre|ciudad  |segmento|
+----------+---------+----------+---------+--------+---------------+-------+------+--------+--------+
|5         |1003     |2025-03-07|Monitor  |2.0     |210            |420.0  |Lucia |Bilbao  |Premium |
|13        |1004     |2025-03-18|Impresora|4.0     |205            |820.0  |Irene |Madrid  |Premium |
|7         |1008     |2025-02-04|Webcam   |6.0     |77             |462.0  |Elena |Bilbao  |Estandar|
|37        |1013     |2025-02-16|Portátil |5.0     |741            |3705.0 |Oscar |Alicante|Premium |
|16        |1014     |2025-02-17|Webcam   |2.0     |78             |156.0  |Diego |Granada |Premium |
|7         |1015     |2025-02-05|Teclado  |5.0     |29             |145.0  |Elena |Bilbao  |Estandar|
|35        |1016     |2025-03-

## 6. Clasifica pedidos con `when` en Alto / Medio / Bajo.

In [18]:
from pyspark.sql.functions import when

# Clasificamos los pedidos según su importe
df_pedidos_clasificados = df_clientes_pedidos.withColumn(
    "categoria_importe",
    when(col("importe") < 100, "Bajo")
    .when((col("importe") >= 100) & (col("importe") < 500), "Medio")
    .otherwise("Alto")
)

# Mostramos una muestra del resultado
print("\n=== Pedidos clasificados por importe ===")
df_pedidos_clasificados.select(
    "id_pedido", "nombre", "producto", "cantidad", "precio_unitario", "importe", "categoria_importe"
).show(10, truncate=False)


=== Pedidos clasificados por importe ===
+---------+-------+-----------+--------+---------------+-------+-----------------+
|id_pedido|nombre |producto   |cantidad|precio_unitario|importe|categoria_importe|
+---------+-------+-----------+--------+---------------+-------+-----------------+
|1001     |Miguel |Ratón      |6.0     |16             |96.0   |Bajo             |
|1002     |Andres |Ratón      |3.0     |19             |57.0   |Bajo             |
|1003     |Lucia  |Monitor    |2.0     |210            |420.0  |Medio            |
|1004     |Irene  |Impresora  |4.0     |205            |820.0  |Alto             |
|1007     |Laura  |Ratón      |1.0     |23             |23.0   |Bajo             |
|1008     |Elena  |Webcam     |6.0     |77             |462.0  |Medio            |
|1009     |Jorge  |Teclado    |1.0     |38             |38.0   |Bajo             |
|1010     |Beatriz|Webcam     |1.0     |78             |78.0   |Bajo             |
|1011     |Gema   |Auriculares|2.0     |32   

## 7. Calcula agregaciones por ciudad y segmento.

In [21]:
from pyspark.sql.functions import count, sum, avg

# Calculamos métricas generales sobre los pedidos clasificados
print("\n=== Métricas generales ===")

df_pedidos_clasificados.select(
    count("id_pedido").alias("numero_pedidos"),
    sum("importe").alias("ingresos_totales"),
    avg("importe").alias("importe_medio")
).show()

# Calculamos métricas agrupadas por segmento
print("\n=== Métricas por segmento ===")

df_pedidos_clasificados.groupBy("segmento").agg(
    count("id_pedido").alias("numero_pedidos"),
    sum("importe").alias("ingresos_totales"),
    avg("importe").alias("importe_medio")
).orderBy("ingresos_totales", ascending=False).show()

# Calculamos un resumen agrupado por ciudad y segmento
print("\n=== Resumen por ciudad y segmento ===")

df_pedidos_clasificados.groupBy("ciudad", "segmento").agg(
    count("id_pedido").alias("numero_pedidos"),
    sum("importe").alias("ingresos_totales"),
    avg("importe").alias("importe_medio")
).orderBy("ciudad", "segmento").show(truncate=False)


=== Métricas generales ===
+--------------+----------------+----------------+
|numero_pedidos|ingresos_totales|   importe_medio|
+--------------+----------------+----------------+
|           106|         61725.0|582.311320754717|
+--------------+----------------+----------------+


=== Métricas por segmento ===
+--------+--------------+----------------+-----------------+
|segmento|numero_pedidos|ingresos_totales|    importe_medio|
+--------+--------------+----------------+-----------------+
|Estandar|            63|         34022.0| 540.031746031746|
| Premium|            43|         27703.0|644.2558139534884|
+--------+--------------+----------------+-----------------+


=== Resumen por ciudad y segmento ===
+----------+--------+--------------+----------------+------------------+
|ciudad    |segmento|numero_pedidos|ingresos_totales|importe_medio     |
+----------+--------+--------------+----------------+------------------+
|Alicante  |Estandar|10            |9113.0          |911.3  

## 8. Crea una vista temporal y haz una consulta SQL.

In [ ]:
# TODO

## 9. Usa `sample()` y `randomSplit()`.

In [ ]:
# TODO

## 10. Guarda el resultado en Parquet en `/opt/spark-apps/salida/resultado_parquet` y léelo de nuevo.

In [ ]:
# TODO

## 11. Responde brevemente en Markdown:
- ¿Qué ventaja tiene usar join?
- ¿Qué diferencia hay entre sample y randomSplit?
- ¿Qué pedido se pierde en el inner join y por qué?

## 12. Cierra la sesión de Spark.

In [ ]:
spark.stop()